<a href="https://colab.research.google.com/github/SebaAcunaC/YOLO11-Vigilancia-Urbana-USACH/blob/main/Semana2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# PROYECTO DE FIN DE GRADO - USACH
# Master Notebook: Benchmarking Multimodelo y Optimización Multiresolución
# Autor: Sebastian [Apellido]
# ==============================================================================

# 1. SETUP DE REPRODUCIBILIDAD
# Instalamos Ultralytics para YOLO y Torchvision para modelos Legacy (SSD/R-CNN)
!pip install ultralytics roboflow torchvision -q

import os
import yaml
import torch
import time
import pandas as pd
from ultralytics import YOLO
from roboflow import Roboflow
from torchvision.models.detection import ssd300_vgg16, fasterrcnn_resnet50_fpn

# 2. CARGA DE DATASET (RECONOCIMIENTO DE PERSONAS)
# Requerimiento: Mismo dataset para todos los modelos
api_key = input("Introduce tu API Key de Roboflow: ")
rf = Roboflow(api_key=api_key)
project = rf.workspace("sebastianacua").project("person-hgivm-vyoba")
dataset = project.version(1).download("yolov8")

# 3. CARGA DE MODELOS (COMPARATIVA SOLICITADA POR EL PROFESOR)
# Cargamos YOLO11 (Tu optimizado), YOLOv8 (Base) y Modelos Legacy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware detectado: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

model_yolo11 = YOLO("yolo11n.pt") # Estado del arte
model_yolov8 = YOLO("yolov8n.pt") # Versión anterior
# SSD y Faster R-CNN (Modelos Legacy requeridos)
model_ssd = ssd300_vgg16(pretrained=True).to(device).eval()
model_rcnn = fasterrcnn_resnet50_fpn(pretrained=True).to(device).eval()

# 4. SCRIPT DE BENCHMARKING MULTI-RESOLUCIÓN
# Aplicamos el barrido de la Semana 5 para justificar el 'Sweet Spot'
def benchmark_model(model_obj, resolutions=[1088, 640, 320]):
    results = []
    for res in resolutions:
        # Nota: Usamos Stride 32 para optimización de memoria en GPU
        print(f"Evaluando en resolución: {res}p...")
        start_time = time.time()
        # Inferencia en modo stream para proteger la RAM
        metrics = model_obj.predict(source=dataset.location + "/valid/images", imgsz=res, device=device, conf=0.5)

        # Cálculo de métricas de ingeniería
        avg_inference = sum([m.speed['inference'] for m in metrics]) / len(metrics)
        fps = 1000 / avg_inference
        results.append({"Res": res, "FPS": round(fps, 2), "Inference_ms": round(avg_inference, 2)})
    return results

# 5. EJECUCIÓN Y TABLA DE RESULTADOS
print("\nIniciando Comparativa Maestra...")
res_yolo11 = benchmark_model(model_yolo11)
# Aquí puedes agregar el benchmark de los otros modelos para completar tus tablas

# Resumen Final para el Mini-Paper
df_final = pd.DataFrame(res_yolo11)
print("\n--- RESULTADOS FINALES YOLO11 ---")
print(df_final)